In [ ]:
#@title Import Modules
import certifi
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
import urllib3
import urllib
import time
import requests
import http.client
from urllib3 import request
from unicodedata import normalize
from io import StringIO
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
#@title Data from FIFA

def extractFIFA():
  # json ranking, point, nama tim
  url_ranking = "https://api.fifa.com/api/v3/fifarankings/rankings/rankingsbyschedule?rankingScheduleId=FRS_Male_Football_20260119&language=en"
  # json match setiap tim
  url_match = "https://inside.fifa.com/api/live-world-ranking/get-match-window-matches?locale=en&gender=1&rankingType=0"
  header = {
      "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
      "Accept": "application/json",
  }

  ranking = requests.get(url_ranking, headers=header)
  data_ranking = ranking.json()

  matches = requests.get(url_match, headers=header)
  data_match = matches.json()

  country_ranking = []
  for item in data_ranking["Results"]:  #ambil data di url_ranking

    id = item.get("IdTeam")
    ranking = item.get("Rank")

    team = item.get("TeamName", [])
    country = team[0].get("Description")

    code = item.get("IdCountry")

    confederation = item.get("ConfederationName")

    points = item.get("TotalPoints")

    # ambil data di url_match
    match_status = []
    goal_in = 0
    goal_out = 0
    matches_raw = data_match.get("matches", {}).get(id, {}).get("MatchesList", []) #ambil match sesuai id team

    for m in matches_raw:
      home_score = m.get("HomeTeamScore", 0)
      away_score = m.get("AwayTeamScore", 0)
      is_home = str(m.get("HomeTeam", {}).get("IdTeam")) == id

      # status dan goal
      current_team_score = home_score if is_home else away_score
      opponent_score = away_score if is_home else home_score

      goal_in += current_team_score
      goal_out += opponent_score

      if current_team_score > opponent_score:
        status = "Menang"
      elif current_team_score < opponent_score:
        status = "Kalah"
      else:
        status = "Seri"

      match_status.append(status)
    country_ranking.append(
        {
            "id": id,
            "ranking": ranking,
            "nama_negara": country,
            "kode_negara": code,
            "konfederasi": confederation,
            "poin fifa": points,
            "status": match_status,
            "goal masuk": goal_in,
            "goal kemasukan": goal_out,
        }
    )


  ranking_list = pd.DataFrame(country_ranking)
  return(ranking_list
         )

extractFIFA()

,id,ranking,nama_negara,kode_negara,konfederasi,poin fifa,status,goal masuk,goal kemasukan
0,43946,1,France,FRA,UEFA,1877.322731,"[Seri, Menang, Menang, Menang, Menang]",14,5
1,43969,2,Spain,ESP,UEFA,1876.395199,"[Menang, Menang, Seri, Menang, Seri]",13,2
2,43922,3,Argentina,ARG,CONMEBOL,1874.814835,"[Menang, Menang, Menang, Menang, Menang]",16,1
3,43942,4,England,ENG,UEFA,1825.965482,"[Menang, Menang, Menang, Seri, Kalah]",10,2
4,43963,5,Portugal,POR,UEFA,1763.834406,"[Seri, Kalah, Menang, Seri, Menang]",13,5
...,...,...,...,...,...,...,...,...,...
206,43895,207,Bahamas,BAH,CONCACAF,786.816217,"[Kalah, Kalah, Kalah, Kalah, Kalah]",2,20
207,20004,208,British Virgin Islands,VGB,CONCACAF,782.137325,"[Menang, Menang, Seri, Menang, Kalah]",13,6
208,1881967,209,US Virgin Islands,VIR,CONCACAF,779.764266,"[Kalah, Kalah, Kalah, Menang, Kalah]",7,16
209,20002,210,Anguilla,AIA,CONCACAF,760.251852,"[Kalah, Menang, Kalah, Menang, Kalah]",5,13


In [ ]:
def mappingNamaNegaraFifa(ranking_list):
  daftar_team = pd.read_csv(
    "/content/drive/MyDrive/Colab Notebooks/Project DE/pesertaWorldCup.csv"
  )

  mappingfifa = {
    "afrika selatan": "south africa",
    "amerika": "usa",
    "arab saudi": "saudi arabia",
    "belanda": "netherlands",
    "belgia": "belgium",
    "bosnia dan herzegovina": "bosnia and herzegovina",
    "cape verde": "cabo verde",
    "ceko": "czechia",
    "curacao": "curaçao",
    "dr congo": "congo dr",
    "ekuador": "ecuador",
    "inggris": "england",
    "irak": "iraq",
    "iran": "ir iran",
    "irlandia utara": "northern ireland",
    "italia": "italy",
    "jamaika": "jamaica",
    "jepang": "japan",
    "jerman": "germany",
    "kaledonia baru": "new caledonia",
    "kanada": "canada",
    "kolombia": "colombia",
    "korea selatan": "korea republic",
    "kroasia": "croatia",
    "makedonia utara": "north macedonia",
    "maroko": "morocco",
    "meksiko": "mexico",
    "mesir": "egypt",
    "norwegia": "norway",
    "pantai gading": "côte d'ivoire",
    "polandia": "poland",
    "prancis": "france",
    "republik irlandia": "republic of ireland",
    "selandia baru": "new zealand",
    "skotlandia": "scotland",
    "spanyol": "spain",
    "swedia": "sweden",
    "swiss": "switzerland",
    "turki": "türkiye",
    "ukraina": "ukraine",
    "yordania": "jordan",
  }

  ubah_nama = {value: key for key, value in mappingfifa.items()}

  def normalize(text):
    if isinstance(text, float):
        return ""
    return text.lower().strip()

  ranking_list["nama_negara"] = ranking_list["nama_negara"].apply(normalize)
  daftar_team["Nama_Negara"] = daftar_team["Nama_Negara"].apply(normalize)

  ranking_list['nama_negara'] = ranking_list['nama_negara'].replace(ubah_nama)
  return(ranking_list)

mappingNamaNegaraFifa(extractFIFA())


,id,ranking,nama_negara,kode_negara,konfederasi,poin fifa,status,goal masuk,goal kemasukan
0,43946,1,prancis,FRA,UEFA,1877.322731,"[Seri, Menang, Menang, Menang, Menang]",14,5
1,43969,2,spanyol,ESP,UEFA,1876.395199,"[Menang, Menang, Seri, Menang, Seri]",13,2
2,43922,3,argentina,ARG,CONMEBOL,1874.814835,"[Menang, Menang, Menang, Menang, Menang]",16,1
3,43942,4,inggris,ENG,UEFA,1825.965482,"[Menang, Menang, Menang, Seri, Kalah]",10,2
4,43963,5,portugal,POR,UEFA,1763.834406,"[Seri, Kalah, Menang, Seri, Menang]",13,5
...,...,...,...,...,...,...,...,...,...
206,43895,207,bahamas,BAH,CONCACAF,786.816217,"[Kalah, Kalah, Kalah, Kalah, Kalah]",2,20
207,20004,208,british virgin islands,VGB,CONCACAF,782.137325,"[Menang, Menang, Seri, Menang, Kalah]",13,6
208,1881967,209,us virgin islands,VIR,CONCACAF,779.764266,"[Kalah, Kalah, Kalah, Menang, Kalah]",7,16
209,20002,210,anguilla,AIA,CONCACAF,760.251852,"[Kalah, Menang, Kalah, Menang, Kalah]",5,13


In [ ]:
#@title Data From Transfermrkt

def extractMarketValue():
  tabel_mentah = []

  headers = {
      "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
      "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
      "Accept-Language": "en-US,en;q=0.9",
      "Referer": "https://www.google.com/",
      "Connection": "keep-alive",
  }

  for halaman in range(1, 11):
      url_dinamis = f"https://www.transfermarkt.co.id/vereins-statistik/wertvollstenationalmannschaften/marktwertetop?page={halaman}"

      response = requests.get(url_dinamis, headers=headers)
      html_data = StringIO(response.text)
      semua_tabel = pd.read_html(html_data)

      for tabel in semua_tabel:
          if "Negara" in str(tabel.head()) or "Nilai Pasar" in str(tabel.head()):
              tabel_mentah.append(tabel)
              break

      time.sleep(2)
  return(tabel_mentah)

extractMarketValue()

[      #           Negara      Konfederasi       Nilai Pasar        Unnamed: 4
 0   1.0          Prancis             UEFA  Rp27.132,74Mlyr.               NaN
 1   NaN          Prancis              NaN               NaN               NaN
 2   NaN                2          Inggris              UEFA  Rp23.868,47Mlyr.
 3   NaN          Inggris              NaN               NaN               NaN
 4   NaN                3          Spanyol              UEFA  Rp21.153,46Mlyr.
 5   NaN          Spanyol              NaN               NaN               NaN
 6   NaN                4         Portugal              UEFA  Rp17.477,24Mlyr.
 7   NaN         Portugal              NaN               NaN               NaN
 8   NaN                5           Jerman              UEFA  Rp17.068,77Mlyr.
 9   NaN           Jerman              NaN               NaN               NaN
 10  NaN                6           Brasil          CONMEBOL  Rp16.394,36Mlyr.
 11  NaN           Brasil              NaN          

In [ ]:
# cleaning hasil scrapping
def transformMarketvalue(tabel_mentah):
  df_mentah_gabungan = pd.concat(tabel_mentah, ignore_index=True)
  df_mentah_gabungan = df_mentah_gabungan.dropna(how="all").reset_index(drop=True)

  negara = df_mentah_gabungan["Negara"].iloc[1::2].reset_index(drop=True)
  baris_genap = df_mentah_gabungan.iloc[0::2].reset_index(drop=True)


  def cari_harga(row):
      for val in row.values[::-1]:
          if isinstance(val, str) and "Rp" in val:
              return val
      return None

  harga = baris_genap.apply(cari_harga, axis=1)

  marketValueData = pd.DataFrame(
      {
          "nama": negara,
          "Market_Value": harga,
      }
  )

  return(marketValueData)

transformMarketvalue(extractMarketValue())

,nama,Market_Value
0,Prancis,"Rp27.132,74Mlyr."
1,Inggris,"Rp23.868,47Mlyr."
2,Spanyol,"Rp21.153,46Mlyr."
3,Portugal,"Rp17.477,24Mlyr."
4,Jerman,"Rp17.068,77Mlyr."
...,...,...
243,India Britania,None
244,Rhodesia Utara,None
245,Vietnam Utara (-1976),None
246,Mayotte,None


In [ ]:
#Menyamakan Nama Negara di transferMRKT

def mappingNamaNegaraMarketValue(marketValueData):

  daftar_team = pd.read_csv(
      "/content/drive/MyDrive/Colab Notebooks/Project DE/pesertaWorldCup.csv"
  )

  mappingtransfermrkt = {
      "algeria": "aljazair",
      "amerika": "amerika serikat",
      "brazil": "brasil",
      "cape verde": "tanjung verde",
      "curacao": "curaçao",
      "dr congo": "republik demokratik kongo",
      "kaledonia baru": "new caledonia",
      "romania": "rumania",
  }

  ubah_nama = {value: key for key, value in mappingtransfermrkt.items()}

  def normalize(text):
    if isinstance(text, float):
        return ""
    return text.lower().strip()

  marketValueData["nama"] = marketValueData["nama"].apply(normalize)
  daftar_team["Nama_Negara"] = daftar_team["Nama_Negara"].apply(normalize)

  marketValueData['nama'] = marketValueData['nama'].replace(ubah_nama)
  return(marketValueData)

mappingNamaNegaraMarketValue(transformMarketvalue(extractMarketValue()))

NameError: name 'transformMarketvalue' is not defined

In [ ]:
def mergeData(ranking_list, marketValueData):

  daftar_team = pd.read_csv(
      "/content/drive/MyDrive/Colab Notebooks/Project DE/pesertaWorldCup.csv"
  )

  daftar_team['Nama_Negara'] = daftar_team['Nama_Negara'].str.lower()

  df_fifa = pd.merge(
    daftar_team,
    ranking_list,
    left_on="Nama_Negara",
    right_on="nama_negara",
    how="left",
  )

  data_master = pd.merge(
    df_fifa,
    marketValueData,
    left_on="Nama_Negara",
    right_on="nama",
    how="left",
  )

  data_master = data_master.drop(columns=["nama", "nama_negara"])

  return(data_master)

mergeData(mappingNamaNegaraFifa(extractFIFA()), mappingNamaNegaraMarketValue(transformMarketvalue(extractMarketValue())))

,Nama_Negara,id,ranking,kode_negara,konfederasi,poin fifa,status,goal masuk,goal kemasukan,Market_Value
0,afrika selatan,43883,60,RSA,CAF,1429.728396,"[Menang, Kalah, Seri, Kalah, Seri]",6,7,"Rp778,70Mlyr."
1,albania,43932,64,ALB,UEFA,1388.058361,"[Menang, Kalah, Kalah, Kalah, Kalah]",2,6,"Rp1.516,55Mlyr."
2,algeria,43843,28,ALG,CAF,1564.256700,"[Menang, Kalah, Menang, Seri, Menang]",9,2,"Rp4.460,13Mlyr."
3,amerika,43921,16,USA,CONCACAF,1673.126608,"[Menang, Menang, Kalah, Kalah, Menang]",12,11,"Rp6.703,23Mlyr."
4,arab saudi,43835,61,KSA,AFC,1421.427596,"[Kalah, Seri, Kalah, Kalah, Kalah]",2,9,"Rp642,69Mlyr."
...,...,...,...,...,...,...,...,...,...,...
59,turki,43972,22,TUR,UEFA,1599.037670,"[Menang, Seri, Menang, Menang, Menang]",10,2,"Rp8.268,45Mlyr."
60,ukraina,43973,32,UKR,UEFA,1546.877672,"[Kalah, Menang, Kalah, Menang, Menang]",6,7,"Rp4.545,30Mlyr."
61,uruguay,43930,17,URU,CONMEBOL,1673.068603,"[Menang, Seri, Kalah, Seri, Seri]",4,7,"Rp6.923,11Mlyr."
62,uzbekistan,44005,50,UZB,AFC,1465.342534,"[Menang, Seri, Menang, Seri, Kalah]",5,3,"Rp1.483,09Mlyr."


In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

# 1. Define paths and Aiven MySQL credentials
csv_file_path = "/content/drive/MyDrive/Colab Notebooks/Project DE/Data Pemain.csv"
ca_cert_path = "/content/drive/MyDrive/Colab Notebooks/Project DE/ca.pem"

db_host = "mysql-3301030a-worldcuppredict.l.aivencloud.com"
db_port = 16372
db_user = "avnadmin"
db_name = "wcp_db"
target_table = "data_pemain"

# 2. Check file and certificate existence before processing
if not os.path.exists(csv_file_path):
    print(f"❌ Error: The file '{csv_file_path}' was not found. Please check the path.")
elif not os.path.exists(ca_cert_path):
    print(f"❌ Error: The SSL certificate '{ca_cert_path}' was not found. Please check the path.")
else:
    try:
        # 3. Load CSV data into a Pandas DataFrame
        df = pd.read_csv(csv_file_path)

        # 4. FIX: Remove duplicate rows based on 'id' column, keeping only the first occurrence
        initial_count = len(df)
        df.drop_duplicates(subset=['id'], keep='first', inplace=True)
        final_count = len(df)

        print(f"Successfully loaded CSV. Removed {initial_count - final_count} duplicate rows.")
        print(f"Total unique rows to upload: {final_count}")

        # 5. Establish secure connection using SQLAlchemy with SSL
        connection_uri = f"mysql+pymysql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
        db_engine = create_engine(
            connection_uri,
            connect_args={"ssl": {"ca": ca_cert_path}}
        )

        # 6. Pre-create the table structure with a PRIMARY KEY explicitly
        with db_engine.begin() as connection:
            connection.execute(text(f"DROP TABLE IF EXISTS {target_table};"))

            create_table_query = f"""
            CREATE TABLE {target_table} (
                id BIGINT PRIMARY KEY,
                name TEXT,
                age BIGINT,
                number FLOAT,
                position TEXT,
                photo TEXT,
                player_rating FLOAT,
                fifa_country_ranking BIGINT,
                country TEXT
            );
            """
            connection.execute(text(create_table_query))
            print(f"Table structure for '{target_table}' with PRIMARY KEY created successfully.")

        # 7. Upload DataFrame by appending to the pre-created table
        df.to_sql(
            name=target_table,
            con=db_engine,
            if_exists="append",
            index=False
        )
        print(f"\n🚀 Success: Uploaded CSV data to '{target_table}' table in Aiven MySQL!")

    except Exception as e:
        print(f"\n❌ Error occurred during data processing or upload: {e}")

Successfully loaded CSV. Removed 2 duplicate rows.
Total unique rows to upload: 1690
Table structure for 'data_pemain' with PRIMARY KEY created successfully.

🚀 Success: Uploaded CSV data to 'data_pemain' table in Aiven MySQL!


In [ ]:
import os
import re
import pandas as pd
from sqlalchemy import create_engine, text

!pip install pymysql -q

# =====================================================
# KONFIGURASI DATABASE AIVEN
# =====================================================

ca_cert_path = "/content/drive/MyDrive/Colab Notebooks/Project DE/ca.pem"

db_host = "mysql-3301030a-worldcuppredict.l.aivencloud.com"
db_port = 16372
db_user = "avnadmin"
db_name = "wcp_db"

# =====================================================
# FILE YANG AKAN DIUPLOAD
# =====================================================

files_to_upload = {
    "/content/drive/MyDrive/Colab Notebooks/Project DE/Data Pemain.csv": {
        "table_name": "data_pemain",
        "pk_column": "id"
    },
    "/content/drive/MyDrive/Colab Notebooks/Project DE/DataMaster.csv": {
        "table_name": "data_master",
        "pk_column": "id"
    }
}

# =====================================================
# DATABASE ENGINE
# =====================================================

connection_uri = (
    f"mysql+pymysql://{db_user}:{db_password}"
    f"@{db_host}:{db_port}/{db_name}"
)

db_engine = create_engine(
    connection_uri,
    connect_args={
        "ssl": {
            "ca": ca_cert_path
        }
    },
    pool_pre_ping=True,
    pool_recycle=3600
)

# =====================================================
# FUNGSI BANTUAN
# =====================================================

def clean_column_name(column_name):
    """
    Mengubah:
    Nama Pemain -> nama_pemain
    Player-Name -> player_name
    """

    column_name = column_name.strip().lower()

    column_name = re.sub(r'[^a-zA-Z0-9_]', '_', column_name)

    column_name = re.sub(r'_+', '_', column_name)

    return column_name


def get_sql_type(series):
    """
    Menentukan tipe data MySQL berdasarkan dtype pandas
    """

    if pd.api.types.is_integer_dtype(series):
        return "BIGINT"

    elif pd.api.types.is_float_dtype(series):
        return "DOUBLE"

    elif pd.api.types.is_bool_dtype(series):
        return "BOOLEAN"

    elif pd.api.types.is_datetime64_any_dtype(series):
        return "DATETIME"

    else:
        max_len = series.astype(str).str.len().max()

        if pd.isna(max_len):
            max_len = 255

        if max_len <= 255:
            return "VARCHAR(255)"

        return "TEXT"


# =====================================================
# PROSES ETL
# =====================================================

for file_path, config in files_to_upload.items():

    table_name = config["table_name"]
    pk_column = config["pk_column"]

    print("\n" + "=" * 60)
    print(f"📂 Memproses file: {file_path}")

    # -------------------------------
    # CEK FILE
    # -------------------------------

    if not os.path.exists(file_path):
        print(f"❌ File tidak ditemukan")
        continue

    try:

        # -------------------------------
        # LOAD CSV
        # -------------------------------

        df = pd.read_csv(file_path)

        original_columns = list(df.columns)

        # Bersihkan nama kolom
        df.columns = [clean_column_name(col) for col in df.columns]

        pk_column = clean_column_name(pk_column)

        print(f"📋 Jumlah kolom : {len(df.columns)}")
        print(f"📊 Jumlah baris : {len(df)}")

        # -------------------------------
        # VALIDASI PRIMARY KEY
        # -------------------------------

        if pk_column not in df.columns:
            raise ValueError(
                f"Primary Key '{pk_column}' tidak ditemukan.\n"
                f"Kolom tersedia: {list(df.columns)}"
            )

        # -------------------------------
        # HAPUS DUPLIKAT
        # -------------------------------

        before = len(df)

        df.drop_duplicates(
            subset=[pk_column],
            keep="first",
            inplace=True
        )

        after = len(df)

        print(
            f"🧹 Duplikat dihapus : {before - after}"
        )

        # -------------------------------
        # HANDLE NULL VALUE
        # -------------------------------

        df = df.where(pd.notnull(df), None)

        # -------------------------------
        # DETEKSI DATETIME OTOMATIS
        # -------------------------------

        for col in df.columns:

            if df[col].dtype == object:

                try:
                    converted = pd.to_datetime(
                        df[col],
                        errors="raise"
                    )

                    if converted.notna().sum() > len(df) * 0.8:
                        df[col] = converted

                except:
                    pass

        # -------------------------------
        # BUAT DDL TABLE
        # -------------------------------

        column_definitions = []

        for col in df.columns:

            sql_type = get_sql_type(df[col])

            if col == pk_column:

                column_definitions.append(
                    f"`{col}` {sql_type} PRIMARY KEY"
                )

            else:

                column_definitions.append(
                    f"`{col}` {sql_type}"
                )

        create_table_query = f"""
        CREATE TABLE `{table_name}` (
            {', '.join(column_definitions)}
        )
        """

        # -------------------------------
        # CREATE TABLE
        # -------------------------------

        with db_engine.begin() as conn:

            conn.execute(
                text(
                    f"DROP TABLE IF EXISTS `{table_name}`"
                )
            )

            conn.execute(
                text(create_table_query)
            )

        print(f"✅ Tabel {table_name} berhasil dibuat")

        # -------------------------------
        # INSERT DATA
        # -------------------------------

        df.to_sql(
            name=table_name,
            con=db_engine,
            if_exists="append",
            index=False,
            chunksize=1000,
            method="multi"
        )

        print(
            f"🚀 Upload selesai "
            f"({len(df)} baris)"
        )

    except Exception as e:

        print(
            f"❌ Gagal memproses "
            f"'{table_name}'\n{str(e)}"
        )

print("\n🏁 Semua proses selesai.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 3.4 MB/s eta 0:00:00

📂 Memproses file: /content/drive/MyDrive/Colab Notebooks/Project DE/Data Pemain.csv
📋 Jumlah kolom : 9
📊 Jumlah baris : 1692
🧹 Duplikat dihapus : 2


/tmp/ipykernel_19084/501866086.py:185: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted = pd.to_datetime(
/tmp/ipykernel_19084/501866086.py:185: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted = pd.to_datetime(
/tmp/ipykernel_19084/501866086.py:185: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted = pd.to_datetime(
/tmp/ipykernel_19084/501866086.py:185: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted = pd.t

✅ Tabel data_pemain berhasil dibuat
🚀 Upload selesai (1690 baris)

📂 Memproses file: /content/drive/MyDrive/Colab Notebooks/Project DE/DataMaster.csv
📋 Jumlah kolom : 11
📊 Jumlah baris : 64
🧹 Duplikat dihapus : 0


/tmp/ipykernel_19084/501866086.py:185: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted = pd.to_datetime(
/tmp/ipykernel_19084/501866086.py:185: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted = pd.to_datetime(
/tmp/ipykernel_19084/501866086.py:185: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted = pd.to_datetime(
/tmp/ipykernel_19084/501866086.py:185: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  converted = pd.t

✅ Tabel data_master berhasil dibuat
🚀 Upload selesai (64 baris)

🏁 Semua proses selesai.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
